# Week 5 — Implementing Machine Learning Algorithms and Model Evaluation
## Predicting Life Expectancy with Regression Models

This notebook builds a predictive regression workflow from the cleaned Gapminder dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 1. Problem definition
**Target:** life expectancy.

**Features:** year, population, log GDP per capita, and continent.

Regression is appropriate because life expectancy is a continuous numerical target. The `development_group` feature is deliberately excluded because it is derived from GDP per capita and could introduce target leakage.

In [ ]:
df=pd.read_csv('../data/ml_model_dataset.csv')
features=['year','pop','log_gdp_per_capita','continent']
X=df[features]; y=df['lifeexp']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)
print('Train:',X_train.shape,'Test:',X_test.shape)


## 2. Preprocessing
Numerical variables are median-imputed and standardized. The categorical continent variable is one-hot encoded. All preprocessing is inside a scikit-learn Pipeline to prevent test-set information from leaking into training.

In [ ]:
num=['year','pop','log_gdp_per_capita']; cat=['continent']
preprocessor=ColumnTransformer([
 ('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),num),
 ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cat)
])


## 3. Baseline — Linear Regression

In [ ]:
linear=Pipeline([('preprocessor',preprocessor),('model',LinearRegression())])
linear.fit(X_train,y_train)
pred_lr=linear.predict(X_test)
print('MAE:',mean_absolute_error(y_test,pred_lr))
print('RMSE:',np.sqrt(mean_squared_error(y_test,pred_lr)))
print('R2:',r2_score(y_test,pred_lr))


## 4. Random Forest and hyperparameter tuning
GridSearchCV evaluates combinations of tree count, tree depth, minimum leaf size, and feature-selection strategy. Five-fold shuffled cross-validation is used, with RMSE as the optimization metric.

In [ ]:
rf=Pipeline([('preprocessor',preprocessor),('model',RandomForestRegressor(random_state=42,n_jobs=-1))])
param_grid={'model__n_estimators':[100,200],'model__max_depth':[None,10,20],'model__min_samples_leaf':[1,2,4],'model__max_features':['sqrt',1.0]}
cv=KFold(n_splits=5,shuffle=True,random_state=42)
grid=GridSearchCV(rf,param_grid=param_grid,cv=cv,scoring='neg_root_mean_squared_error',n_jobs=-1)
grid.fit(X_train,y_train)
print('Best parameters:',grid.best_params_)
print('Best CV RMSE:',-grid.best_score_)


## 5. Final model evaluation

In [ ]:
best=grid.best_estimator_
pred=best.predict(X_test)
print('MAE:',mean_absolute_error(y_test,pred))
print('RMSE:',np.sqrt(mean_squared_error(y_test,pred)))
print('R2:',r2_score(y_test,pred))


## 6. Cross-validation stability

In [ ]:
scores=-cross_val_score(best,X_train,y_train,cv=cv,scoring='neg_root_mean_squared_error')
print('Fold RMSE:',scores)
print('Mean:',scores.mean(),'Std:',scores.std())


## 7. Interpretation
Compare MAE, RMSE, and R². Lower MAE/RMSE indicate smaller prediction errors, while higher R² indicates more variance explained. The final interpretation must distinguish held-out test performance from cross-validation performance and must not claim that the model is causal.

In [ ]:
plt.figure(figsize=(8,6)); plt.scatter(y_test,pred,alpha=.7)
mn=min(y_test.min(),pred.min()); mx=max(y_test.max(),pred.max())
plt.plot([mn,mx],[mn,mx],'--'); plt.xlabel('Actual'); plt.ylabel('Predicted'); plt.title('Actual vs Predicted Life Expectancy'); plt.show()


In [ ]:
resid=y_test.values-pred
plt.figure(figsize=(8,6)); plt.scatter(pred,resid,alpha=.7); plt.axhline(0,linestyle='--')
plt.xlabel('Predicted Life Expectancy'); plt.ylabel('Residual'); plt.title('Residual Plot'); plt.show()
